In [37]:
import pandas as pd

df = pd.read_parquet('data/medical_consultation/train.parquet')

In [39]:
# 替换df.reward_model.values里每一项中的"\'"为"""

for i in range(len(df.reward_model.values)):
    df.reward_model.values[i] = df.reward_model.values[i].replace("\'", '"')

df.to_parquet("data/medical_consultation/train.parquet")


In [41]:
import json
for i in range(len(df.reward_model.values)):
    json.loads(df.reward_model.values[i])

JSONDecodeError: Expecting ',' delimiter: line 1 column 323 (char 322)

In [43]:
i

229

In [8]:
# 给df的每一行添加extra_info: {"index": i}
df['extra_info'] = [{'index': i, 'split': 'train'} for i in range(len(df))]

# 保存到parquet
df.to_parquet('data/medical_consultation/train.parquet')


In [7]:
df.extra_info.values

array([{'index': 0}, {'index': 1}, {'index': 2}, ..., {'index': 15191},
       {'index': 15192}, {'index': 15193}], dtype=object)

In [17]:
df['prompt'][1]

'[{\'content\': "<|im_start|>user\n你是一位经验丰富的医生，需要通过问诊为患者提供专业的诊断和建议。请仔细倾听患者的描述，提出有针对性的问题，收集足够的信息后再给出诊断和治疗建议。\n\\ 快速指南\n目标:1. 通过有效提问获取关键信息，每一轮提问都要基于上一轮内容修改，即不能询问类似问题。\n2. 综合分析患者情况，给出准确的诊断和合适的治疗建议。 \n\n规则:\n1. 你只能选择其中一个选项回复，不能同时回答问题和给出诊断。\n2.绝对不要重复或询问与之前已问过的相似或相同的问题\n\n回答:\n<answer>如果你认为信息不足，请只提出一个问题，格式为：\n问题: (你的问题)。 </answer> | <answer> 如果你认为已获得足够信息，请只给出诊断和建议，格式为：\n诊断: (患者最可能的疾病或症状)\n建议: (相应的治疗方案或建议)\n </answer>\n\n奖励:\n每提问一次: -0.5\n提问有效（患者能给出答案）: +1.0\n提问无效则不计分\n 重复提问:-1.0\n达到最大交互轮次未给出诊断:-5.0\n诊断和建议都正确: +10.0\n\n\n决定下一步行动:\n总是输出: <think> [你的思考] </think> <answer> [你的回答] </answer> 无额外文本。 严格遵循此格式。<|im_end|>\n<|im_start|>assistant\n<think>", \'role\': \'user\'}]'

In [13]:
def _calculate_lcs(str1: str, str2: str) -> str:
    """
    Calculate the longest common subsequence between two strings.
    """
    # Convert strings to lowercase for case-insensitive comparison
    str1 = str1.lower()
    str2 = str2.lower()
    
    # Get lengths of strings
    m = len(str1)
    n = len(str2)
    
    # Initialize LCS matrix
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    # Fill dp table
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i-1] == str2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
                
    # Backtrack to find LCS string
    lcs = []
    i, j = m, n
    while i > 0 and j > 0:
        if str1[i-1] == str2[j-1]:
            lcs.append(str1[i-1])
            i -= 1
            j -= 1
        elif dp[i-1][j] > dp[i][j-1]:
            i -= 1
        else:
            j -= 1
            
    return ''.join(reversed(lcs))

In [14]:
_calculate_lcs("小儿麻痹症", "小儿")

'小儿'